# Batch 06 — Exp03: Deep-Dive Deteksi Biner Overload pada `v4_self_reported_rating`

Notebook mandiri untuk pendalaman deteksi biner beban tinggi vs rendah pada dataset **`v4_self_reported_rating`** (`window_15s_step_5s/good_acceptable.csv`, 1.693 window, 126 episode, 17 peserta) tanpa menambah data baru.

**Tiga Definisi Target Biner yang Diuji:**
1. **`extreme_1_3_vs_7_10`**: Low (`1–3`) vs High (`7–10`), mengeluarkan kelas tengah ambigu `4–6` (89 episode, 1.219 window).
2. **`full_1_5_vs_6_10`**: Seluruh 100% data V4 — Low/Manageable (`1–5`) vs Elevated/High (`6–10`) (126 episode, 1.693 window).
3. **`full_1_6_vs_7_10`**: Seluruh 100% data V4 — Non-Overload (`1–6`) vs Severe Overload (`7–10`) (126 episode, 1.693 window), langsung sesuai pemicu intervensi Adaptive-IDE.

**Protokol Evaluasi Ketat (Bebas Kebocoran):**
- **Leave-One-Participant-Out (LOPO) CV** pada seluruh 17 peserta (setiap peserta dievaluasi 100% out-of-sample), dilengkapi **bootstrap 95% CI** tingkat episode/peserta dan uji signifikansi Mann-Whitney $U$.
- **Nested inner-CV** untuk pemilihan hiperparameter (`C`) dan ambang keputusan (*Youden's J* / optimal Balanced Accuracy) hanya dari fold training — tidak pernah di-tune pada peserta uji.
- **Evaluasi silang pada Held-Out Test beku (4 peserta dari Exp02)** sebagai verifikasi konsistensi.
- **Ablasi grup fitur (`HRV`, `Eye`, `HeadPose`, `BaselineRatio`, `GazeRatio`)**, inspeksi koefisien terstandardisasi, serta **audit faktor perancu** (*confounding check* terhadap waktu sesi dan kesulitan protokol).


In [1]:
# Cell 1: Konfigurasi Lingkungan, Seed, dan Verifikasi SHA-256 V4
import hashlib
import json
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import platform
import random
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
if "ipykernel" not in os.sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IN_COLAB = "google.colab" in os.sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/Penelitian-Adaptive-IDE")
    DATASET_ROOT = PROJECT_ROOT / "models/batch-06/dataset"
else:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd.parent.parent.parent,
        Path(r"e:\Skripsi\ngodong\Adaptive-IDE"),
    ]
    PROJECT_ROOT = next((p for p in candidates if (p / "models/batch-06/dataset").exists()), cwd)
    DATASET_ROOT = PROJECT_ROOT / "models/batch-06/dataset"

V4_DIR = DATASET_ROOT / "v4_self_reported_rating"
EXP02_SPLIT_PATH = PROJECT_ROOT / "models/batch-06/outputs/exp02_v4_self_reported_rating/participant_splits_v4.csv"
OUTPUT_DIR = PROJECT_ROOT / "models/batch-06/outputs/exp03_v4_binary_overload"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    "window_15s_step_5s/good_acceptable.csv",
    "window_15s_step_5s/window_assignments.csv",
    "episode_ratings.csv",
]
file_hashes = {}
for rel in REQUIRED_FILES:
    p = V4_DIR / rel
    assert p.exists(), f"File wajib tidak ditemukan: {p}"
    file_hashes[rel] = hashlib.sha256(p.read_bytes()).hexdigest()

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"V4_DIR       : {V4_DIR}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"Python {platform.python_version()} | SHA-256 dataset v4 terverifikasi (3/3 file).")


PROJECT_ROOT : E:\Skripsi\ngodong\Adaptive-IDE
V4_DIR       : E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\dataset\v4_self_reported_rating
OUTPUT_DIR   : E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\outputs\exp03_v4_binary_overload
Python 3.10.11 | SHA-256 dataset v4 terverifikasi (3/3 file).


In [2]:
# Cell 2: Muat Data V4, Definisikan 3 Skema Biner & Profil Fitur
HRV2 = ["rmssd_ms", "sdnn_ms"]
EYE7 = [
    "blink_rate_per_minute",
    "ear_left_mean",
    "ear_left_std",
    "ear_right_mean",
    "ear_right_std",
    "iris_size_estimate_mean",
    "iris_size_estimate_std",
]
CORE9 = HRV2 + EYE7
CORE10 = CORE9 + ["iris_size_delta_mean"]
HEAD4 = ["head_pitch_mean", "head_pitch_std", "head_yaw_mean", "head_yaw_std"]
BASE1 = ["rmssd_baseline_ratio"]
GAZE2 = ["on_screen_ratio", "valid_gaze_ratio"]

FEATURE_PROFILES = {
    "core10": CORE10,
    "core9": CORE9,
    "core9_head4": CORE9 + HEAD4,
    "core9_head4_base": CORE9 + HEAD4 + BASE1,
    "core9_head4_base_gaze": CORE9 + HEAD4 + BASE1 + GAZE2,
    # Grup tunggal untuk ablasi atribusi sinyal
    "only_hrv2": HRV2,
    "only_eye7": EYE7,
    "only_head4": HEAD4,
    "only_base1": BASE1,
    "only_head4_base1": HEAD4 + BASE1,
}

raw_df = pd.read_csv(V4_DIR / "window_15s_step_5s/good_acceptable.csv")
wa_df = pd.read_csv(V4_DIR / "window_15s_step_5s/window_assignments.csv")

assert raw_df["window_id"].is_unique
df = raw_df.merge(
    wa_df[["window_id", "episode_id", "difficulty_rating"]].rename(columns={"difficulty_rating": "wa_rating"}),
    on="window_id",
    how="left",
    validate="1:1",
)
assert df["episode_id"].notna().all()
assert (df["difficulty_rating"] == df["wa_rating"]).all()
df = df.drop(columns=["wa_rating"])

df["window_start_dt"] = pd.to_datetime(df["window_start"])
df["session_elapsed_min"] = (
    df["window_start_dt"] - df.groupby("session_id")["window_start_dt"].transform("min")
).dt.total_seconds() / 60.0
df = df.sort_values(["session_id", "window_start_dt", "window_id"]).reset_index(drop=True)

# Pakai split Exp02 yang sama jika tersedia (4 peserta Held-Out Test beku: Athaya, Baithar, Sullivan, Tio)
if EXP02_SPLIT_PATH.exists():
    exp02_splits = pd.read_csv(EXP02_SPLIT_PATH)
    df = df.merge(exp02_splits, on="participant_id", how="left", validate="m:1")
else:
    test_people = {"Athaya", "Baithar", "Sullivan", "Tio"}
    df["split"] = np.where(df["participant_id"].isin(test_people), "Test", "Dev")

BINARY_TASKS = {
    "extreme_1_3_vs_7_10": {
        "desc": "Low (1–3) vs High (7–10), drop 4–6",
        "low_vals": {1, 2, 3},
        "high_vals": {7, 8, 9, 10},
    },
    "full_1_5_vs_6_10": {
        "desc": "Full Data: Low (1–5) vs High (6–10)",
        "low_vals": {1, 2, 3, 4, 5},
        "high_vals": {6, 7, 8, 9, 10},
    },
    "full_1_6_vs_7_10": {
        "desc": "Full Data: Non-Overload (1–6) vs Severe Overload (7–10)",
        "low_vals": {1, 2, 3, 4, 5, 6},
        "high_vals": {7, 8, 9, 10},
    },
}

# Precompute statistik kausal (expanding median & IQR dari window sebelumnya dalam sesi yang sama)
# 100% kausal & independen terhadap split (hanya memakai baris sebelumnya dari sesi itu sendiri;
# posisi < 3 dan fallback IQR==0 tetap memakai g_med_/g_iqr_ milik fitting fold masing-masing).
ALL_SENSOR_COLS = sorted({c for cols in FEATURE_PROFILES.values() for c in cols})
CAUSAL_MED = {}
CAUSAL_IQR = {}
CAUSAL_POS = np.zeros(len(df), dtype=int)
for col in ALL_SENSOR_COLS:
    c_med = np.full(len(df), np.nan, dtype=float)
    c_iqr = np.full(len(df), np.nan, dtype=float)
    vals = df[col].to_numpy(dtype=float)
    for sid, idx in df.groupby("session_id").indices.items():
        # df sudah terurut kronologis per session_id
        for pos, ri in enumerate(idx):
            CAUSAL_POS[ri] = pos
            if pos >= 3:
                hist = vals[idx[:pos]]
                hist = hist[np.isfinite(hist)]
                if len(hist) >= 3:
                    c_med[ri] = np.median(hist)
                    q75, q25 = np.percentile(hist, [75, 25])
                    c_iqr[ri] = q75 - q25
    CAUSAL_MED[col] = c_med
    CAUSAL_IQR[col] = c_iqr
df["_row_idx"] = np.arange(len(df), dtype=int)

print("Ringkasan 3 Definisi Target Biner V4:")
for tname, cfg in BINARY_TASKS.items():
    sub = df[df["difficulty_rating"].isin(cfg["low_vals"] | cfg["high_vals"])]
    lo = sub[sub["difficulty_rating"].isin(cfg["low_vals"])]
    hi = sub[sub["difficulty_rating"].isin(cfg["high_vals"])]
    print(
        f"  {tname:20s}: total={len(sub):4d}w/{sub['episode_id'].nunique():3d}ep/{sub['participant_id'].nunique():2d}p | "
        f"Low={len(lo):3d}w/{lo['episode_id'].nunique():3d}ep/{lo['participant_id'].nunique():2d}p | "
        f"High={len(hi):3d}w/{hi['episode_id'].nunique():3d}ep/{hi['participant_id'].nunique():2d}p"
    )


Ringkasan 3 Definisi Target Biner V4:
  extreme_1_3_vs_7_10 : total=1219w/ 89ep/17p | Low=512w/ 66ep/14p | High=707w/ 23ep/12p
  full_1_5_vs_6_10    : total=1693w/126ep/17p | Low=785w/ 92ep/15p | High=908w/ 34ep/17p
  full_1_6_vs_7_10    : total=1693w/126ep/17p | Low=986w/103ep/15p | High=707w/ 23ep/12p


In [3]:
# Cell 3: Preprocessor, Bobot, Model dengan Nested Inner-CV Tuning, & Metrik + Bootstrap CI
class RobustPreprocessor:
    def __init__(self, cols: list, mode: str = "global_robust"):
        self.cols = cols
        self.mode = mode
        self.g_med_ = None
        self.g_iqr_ = None

    def fit(self, fit_df: pd.DataFrame):
        vals = fit_df[self.cols].to_numpy(dtype=float)
        self.g_med_ = np.nanmedian(vals, axis=0)
        filled = np.where(np.isfinite(vals), vals, self.g_med_)
        q75, q25 = np.percentile(filled, [75, 25], axis=0)
        iqr = np.where((q75 - q25) > 1e-9, q75 - q25, 1.0)
        self.g_iqr_ = iqr
        return self

    def transform(self, target_df: pd.DataFrame) -> np.ndarray:
        vals = target_df[self.cols].to_numpy(dtype=float)
        vals = np.where(np.isfinite(vals), vals, self.g_med_)
        if self.mode == "global_robust":
            return np.clip((vals - self.g_med_) / self.g_iqr_, -10.0, 10.0)
        out = np.zeros_like(vals, dtype=float)
        if self.mode == "causal_session_robust":
            ridx = target_df["_row_idx"].to_numpy(dtype=int)
            out = (vals - self.g_med_) / self.g_iqr_
            use_c = CAUSAL_POS[ridx] >= 3
            if use_c.any():
                c_med = np.column_stack([CAUSAL_MED[c][ridx] for c in self.cols])
                c_iqr = np.column_stack([CAUSAL_IQR[c][ridx] for c in self.cols])
                valid_m = np.isfinite(c_med) & use_c[:, None]
                eff_med = np.where(valid_m, c_med, self.g_med_)
                eff_iqr = np.where(valid_m & np.isfinite(c_iqr) & (c_iqr > 1e-9), c_iqr, self.g_iqr_)
                out = (vals - eff_med) / eff_iqr
            return np.clip(out, -10.0, 10.0)
        if self.mode == "unsupervised_session_robust":
            for sid, idx in target_df.groupby("session_id").indices.items():
                blk = vals[idx]
                if len(blk) < 3:
                    out[idx] = (blk - self.g_med_) / self.g_iqr_
                else:
                    med = np.median(blk, axis=0)
                    q75, q25 = np.percentile(blk, [75, 25], axis=0)
                    iqr = np.where((q75 - q25) > 1e-9, q75 - q25, self.g_iqr_)
                    out[idx] = (blk - med) / iqr
            return np.clip(out, -10.0, 10.0)
        raise ValueError(self.mode)


def episode_balanced_weights(sub_df: pd.DataFrame, y_bin: np.ndarray, clip_max: float = 5.0) -> np.ndarray:
    """Setarakan kontribusi kelas dan episode dalam fitting fold."""
    n = len(y_bin)
    w = np.ones(n, dtype=float)
    for cls in (0, 1):
        mask = y_bin == cls
        if not mask.any():
            continue
        g = sub_df.loc[mask]
        n_ep = g["episode_id"].nunique()
        ep_sz = g.groupby("episode_id")["window_id"].transform("count").to_numpy(dtype=float)
        wc = (n / (2.0 * n_ep)) / ep_sz
        wc = np.minimum(wc, clip_max)
        wc = wc * ((n / 2.0) / wc.sum())
        w[mask] = wc
    return w


def select_threshold_from_oof(y_true: np.ndarray, probs: np.ndarray) -> float:
    """Pilih ambang keputusan optimal (Youden's J / Balanced Accuracy) HANYA dari data training/inner-OOF."""
    fpr, tpr, thresholds = roc_curve(y_true, probs)
    j = tpr - fpr
    best_i = int(np.argmax(j))
    thr = float(thresholds[best_i])
    return float(np.clip(thr, 0.15, 0.85))


def fit_predict_binary(model_name: str, X_tr, y_tr, w_tr, groups_tr, X_eval):
    if model_name == "Dummy":
        clf = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
        return clf.predict_proba(X_eval)[:, 1], 0.50

    if model_name == "LogReg_L2":
        clf = LogisticRegression(C=0.5, max_iter=5000, random_state=SEED)
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        return clf.predict_proba(X_eval)[:, 1], 0.50

    if model_name == "LogReg_NestedC":
        # Pilih C terbaik + ambang optimal lewat 4-fold inner GroupKFold di dalam fitting fold
        n_g = len(np.unique(groups_tr))
        n_sp = min(4, n_g)
        gkf = GroupKFold(n_splits=n_sp)
        best_c, best_auc, best_oof = 0.5, -1.0, None
        for c_val in (0.02, 0.1, 0.5, 2.0):
            oof_p = np.zeros(len(y_tr), dtype=float)
            valid_folds = True
            for itr, iva in gkf.split(X_tr, y_tr, groups_tr):
                if len(np.unique(y_tr[itr])) < 2:
                    valid_folds = False
                    break
                m = LogisticRegression(C=c_val, max_iter=5000, random_state=SEED)
                m.fit(X_tr[itr], y_tr[itr], sample_weight=w_tr[itr])
                oof_p[iva] = m.predict_proba(X_tr[iva])[:, 1]
            if valid_folds and len(np.unique(y_tr)) == 2:
                auc = roc_auc_score(y_tr, oof_p)
                if auc > best_auc:
                    best_auc, best_c, best_oof = auc, c_val, oof_p
        thr = select_threshold_from_oof(y_tr, best_oof) if best_oof is not None else 0.50
        clf = LogisticRegression(C=best_c, max_iter=5000, random_state=SEED)
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        return clf.predict_proba(X_eval)[:, 1], thr

    if model_name == "LogReg_L1":
        clf = LogisticRegression(penalty="l1", solver="liblinear", C=0.5, max_iter=2000, random_state=SEED)
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        return clf.predict_proba(X_eval)[:, 1], 0.50

    if model_name == "ExtraTrees_Shallow":
        clf = ExtraTreesClassifier(
            n_estimators=100, max_depth=4, min_samples_leaf=8, max_features=0.7, random_state=SEED, n_jobs=1
        )
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        return clf.predict_proba(X_eval)[:, 1], 0.50

    if model_name == "Blend_LR_ET":
        lr = LogisticRegression(C=0.5, max_iter=5000, random_state=SEED).fit(X_tr, y_tr, sample_weight=w_tr)
        et = ExtraTreesClassifier(
            n_estimators=100, max_depth=4, min_samples_leaf=8, max_features=0.7, random_state=SEED, n_jobs=1
        ).fit(X_tr, y_tr, sample_weight=w_tr)
        p = 0.65 * lr.predict_proba(X_eval)[:, 1] + 0.35 * et.predict_proba(X_eval)[:, 1]
        return p, 0.50

    raise ValueError(model_name)


def bootstrap_ep_auc_ci(ep_y: np.ndarray, ep_p: np.ndarray, ep_pids: np.ndarray, n_boot: int = 1000, seed: int = SEED):
    """Cluster-bootstrap 95% CI pada tingkat peserta (meresample peserta dengan pengembalian)."""
    rng = np.random.RandomState(seed)
    uniq_p = np.unique(ep_pids)
    p_to_idx = {p: np.where(ep_pids == p)[0] for p in uniq_p}
    aucs = []
    for _ in range(n_boot):
        sampled_p = rng.choice(uniq_p, size=len(uniq_p), replace=True)
        idx = np.concatenate([p_to_idx[p] for p in sampled_p])
        if len(np.unique(ep_y[idx])) < 2:
            continue
        aucs.append(roc_auc_score(ep_y[idx], ep_p[idx]))
    if len(aucs) < 50:
        return np.nan, np.nan
    return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))


def evaluate_binary_full(y_true, probs, thresholds, episode_ids, participant_ids, compute_ci: bool = False) -> dict:
    probs = np.clip(probs, 1e-6, 1.0 - 1e-6)
    pred_w = (probs >= thresholds).astype(int)
    w_auc = float(roc_auc_score(y_true, probs))
    w_ap = float(average_precision_score(y_true, probs))
    w_acc = float(accuracy_score(y_true, pred_w))
    w_bal = float(balanced_accuracy_score(y_true, pred_w))
    w_mf1 = float(f1_score(y_true, pred_w, average="macro", zero_division=0))
    w_sens = float(recall_score(y_true, pred_w, pos_label=1, zero_division=0))
    w_spec = float(recall_score(y_true, pred_w, pos_label=0, zero_division=0))
    w_brier = float(brier_score_loss(y_true, probs))

    ep_df = pd.DataFrame({
        "ep": episode_ids, "pid": participant_ids, "y": y_true, "p": probs, "thr": thresholds
    }).groupby(["ep", "pid"], as_index=False).mean()
    ep_y = ep_df["y"].round().astype(int).to_numpy()
    ep_p = ep_df["p"].to_numpy()
    ep_thr = ep_df["thr"].to_numpy()
    ep_pred = (ep_p >= ep_thr).astype(int)

    ep_auc = float(roc_auc_score(ep_y, ep_p))
    ep_ap = float(average_precision_score(ep_y, ep_p))
    ep_acc = float(accuracy_score(ep_y, ep_pred))
    ep_bal = float(balanced_accuracy_score(ep_y, ep_pred))
    ep_mf1 = float(f1_score(ep_y, ep_pred, average="macro", zero_division=0))
    ep_sens = float(recall_score(ep_y, ep_pred, pos_label=1, zero_division=0))
    ep_spec = float(recall_score(ep_y, ep_pred, pos_label=0, zero_division=0))
    ep_brier = float(brier_score_loss(ep_y, ep_p))
    # Uji Mann-Whitney U satu sisi (prob High > prob Low pada episode independen)
    mw_p = float(stats.mannwhitneyu(ep_p[ep_y == 1], ep_p[ep_y == 0], alternative="greater").pvalue)

    res = {
        "n_windows": int(len(y_true)),
        "n_episodes": int(len(ep_y)),
        "win_auc": w_auc, "win_ap": w_ap, "win_acc": w_acc, "win_bal_acc": w_bal,
        "win_macro_f1": w_mf1, "win_sens": w_sens, "win_spec": w_spec, "win_brier": w_brier,
        "ep_auc": ep_auc, "ep_ap": ep_ap, "ep_acc": ep_acc, "ep_bal_acc": ep_bal,
        "ep_macro_f1": ep_mf1, "ep_sens": ep_sens, "ep_spec": ep_spec, "ep_brier": ep_brier,
        "ep_mwu_pvalue": mw_p,
    }
    if compute_ci:
        ci_lo, ci_hi = bootstrap_ep_auc_ci(ep_y, ep_p, ep_df["pid"].to_numpy())
        res["ep_auc_ci95_lo"] = ci_lo
        res["ep_auc_ci95_hi"] = ci_hi
    return res


In [4]:
# Cell 4: Benchmark LOPO Lengkap (17 Peserta) pada 3 Definisi Biner + Ablasi Fitur & Model
t0 = time.time()
logo = LeaveOneGroupOut()

MAIN_FEATS = ["core10", "core9", "core9_head4", "core9_head4_base", "core9_head4_base_gaze"]
GROUP_FEATS = ["only_hrv2", "only_eye7", "only_head4", "only_base1", "only_head4_base1"]
MODELS = ["Dummy", "LogReg_L2", "LogReg_NestedC", "LogReg_L1", "ExtraTrees_Shallow", "Blend_LR_ET"]
NORMS = ["global_robust", "causal_session_robust", "unsupervised_session_robust"]

lopo_rows = []
oof_store = {}  # simpan OOF terbaik untuk visualisasi & audit perancu

for task_name, tcfg in BINARY_TASKS.items():
    sub = df[df["difficulty_rating"].isin(tcfg["low_vals"] | tcfg["high_vals"])].reset_index(drop=True)
    y = sub["difficulty_rating"].isin(tcfg["high_vals"]).astype(int).to_numpy()
    groups = sub["participant_id"].to_numpy()
    eps = sub["episode_id"].to_numpy()
    splits = list(logo.split(sub, y, groups))

    # Cache normalisasi per (fold, feat, norm)
    fcache = {}
    def get_xy(fi, tr, va, feat, norm):
        k = (fi, feat, norm)
        if k not in fcache:
            prep = RobustPreprocessor(FEATURE_PROFILES[feat], norm).fit(sub.iloc[tr])
            fcache[k] = (prep.transform(sub.iloc[tr]), prep.transform(sub.iloc[va]))
        return fcache[k]

    # Grid utama: LogReg_L2 pada semua MAIN_FEATS x NORMS; model lain pada 3 profil inti x 2 normalisasi operasional
    specs = [("Dummy", "core10", "global_robust")]
    for feat in MAIN_FEATS:
        for norm in NORMS:
            specs.append(("LogReg_L2", feat, norm))
    for feat in ("core9", "core9_head4", "core9_head4_base"):
        for norm in ("global_robust", "causal_session_robust"):
            for m in ("LogReg_NestedC", "LogReg_L1", "ExtraTrees_Shallow", "Blend_LR_ET"):
                specs.append((m, feat, norm))
    # Ablasi grup fitur tunggal pada LogReg_L2 + global_robust & causal_session_robust
    for feat in GROUP_FEATS:
        for norm in ("global_robust", "causal_session_robust"):
            specs.append(("LogReg_L2", feat, norm))

    for m_name, feat, norm in specs:
        oof_p = np.zeros(len(y), dtype=float)
        oof_thr = np.full(len(y), 0.50, dtype=float)
        for fi, (tr, va) in enumerate(splits):
            X_tr, X_va = get_xy(fi, tr, va, feat, norm)
            w_tr = episode_balanced_weights(sub.iloc[tr], y[tr])
            p_va, thr = fit_predict_binary(m_name, X_tr, y[tr], w_tr, groups[tr], X_va)
            oof_p[va] = p_va
            oof_thr[va] = thr

        do_ci = (m_name in ("LogReg_L2", "LogReg_NestedC", "Blend_LR_ET") and feat in ("core9", "core9_head4_base", "core9_head4_base_gaze") and norm in ("global_robust", "causal_session_robust"))
        met = evaluate_binary_full(y, oof_p, oof_thr, eps, groups, compute_ci=do_ci)
        cid = f"{m_name}|{feat}|{norm}"
        lopo_rows.append({
            "task": task_name, "candidate_id": cid, "model": m_name,
            "features": feat, "normalization": norm,
            "is_single_group_ablation": feat in GROUP_FEATS,
            **met,
        })
        oof_store[(task_name, cid)] = (sub, y, oof_p, oof_thr)

lopo_df = pd.DataFrame(lopo_rows).sort_values(["task", "ep_auc", "win_auc"], ascending=[True, False, False]).reset_index(drop=True)
lopo_df.to_csv(OUTPUT_DIR / "lopo_all_tasks_leaderboard.csv", index=False)

print(f"Selesai LOPO 17-peserta ({len(lopo_df)} evaluasi) dalam {time.time()-t0:.1f} detik.")
for tname in BINARY_TASKS:
    sub_lb = lopo_df[(lopo_df["task"] == tname) & (~lopo_df["is_single_group_ablation"]) & (lopo_df["normalization"] != "unsupervised_session_robust")]
    print(f"\n=== TOP 5 LOPO OPERASIONAL: {tname} ===")
    cols_show = ["candidate_id", "win_auc", "win_bal_acc", "ep_auc", "ep_auc_ci95_lo", "ep_auc_ci95_hi", "ep_bal_acc", "ep_acc", "ep_mwu_pvalue"]
    print(sub_lb[cols_show].head(5).to_string(index=False))


Selesai LOPO 17-peserta (150 evaluasi) dalam 155.4 detik.

=== TOP 5 LOPO OPERASIONAL: extreme_1_3_vs_7_10 ===
                                 candidate_id  win_auc  win_bal_acc   ep_auc  ep_auc_ci95_lo  ep_auc_ci95_hi  ep_bal_acc   ep_acc  ep_mwu_pvalue
     LogReg_L1|core9_head4_base|global_robust 0.752050     0.656946 0.783926             NaN             NaN    0.674572 0.685393       0.000027
LogReg_NestedC|core9_head4_base|global_robust 0.742304     0.603302 0.779315        0.610408        0.925385    0.600791 0.617978       0.000036
     LogReg_L2|core9_head4_base|global_robust 0.743425     0.646743 0.777339        0.610032        0.924007    0.682148 0.696629       0.000041
   Blend_LR_ET|core9_head4_base|global_robust 0.715957     0.620544 0.756258        0.572825        0.911544    0.651845 0.651685       0.000136
                LogReg_L2|core9|global_robust 0.670309     0.632500 0.752964        0.606233        0.909043    0.679183 0.629213       0.000163

=== TOP 5 LOPO OPE

In [5]:
# Cell 5: Verifikasi Silang pada Held-Out Test Beku (Exp02: 13 Dev -> 4 Test) + Audit Koefisien & Perancu
# Pilih kandidat pemenang per task HANYA dari LOPO pada 13 peserta Dev (tanpa melihat 4 peserta Test),
# lalu evaluasi 1x pada 4 peserta Held-Out Test (Athaya, Baithar, Sullivan, Tio).
dev_test_summary = {}

for task_name, tcfg in BINARY_TASKS.items():
    sub = df[df["difficulty_rating"].isin(tcfg["low_vals"] | tcfg["high_vals"])].reset_index(drop=True)
    dev_sub = sub[sub["split"] == "Dev"].reset_index(drop=True)
    te_sub = sub[sub["split"] == "Test"].reset_index(drop=True)
    y_dev = dev_sub["difficulty_rating"].isin(tcfg["high_vals"]).astype(int).to_numpy()
    y_te = te_sub["difficulty_rating"].isin(tcfg["high_vals"]).astype(int).to_numpy()

    # LOPO khusus di dalam 13 peserta Dev untuk mengunci pemenang secara ketat
    dev_splits = list(LeaveOneGroupOut().split(dev_sub, y_dev, dev_sub["participant_id"].to_numpy()))
    dcache = {}
    def get_dev_xy(fi, tr, va, feat, norm):
        k = (fi, feat, norm)
        if k not in dcache:
            prep = RobustPreprocessor(FEATURE_PROFILES[feat], norm).fit(dev_sub.iloc[tr])
            dcache[k] = (prep.transform(dev_sub.iloc[tr]), prep.transform(dev_sub.iloc[va]))
        return dcache[k]

    best_dev_row = None
    best_dev_auc = -1.0
    for feat in ("core9", "core9_head4", "core9_head4_base", "core9_head4_base_gaze"):
        for norm in ("global_robust", "causal_session_robust"):
            for m_name in ("LogReg_L2", "LogReg_NestedC", "ExtraTrees_Shallow", "Blend_LR_ET"):
                oof_p = np.zeros(len(y_dev), dtype=float)
                oof_thr = np.full(len(y_dev), 0.50, dtype=float)
                for fi, (tr, va) in enumerate(dev_splits):
                    X_tr, X_va = get_dev_xy(fi, tr, va, feat, norm)
                    w_tr = episode_balanced_weights(dev_sub.iloc[tr], y_dev[tr])
                    p_va, thr = fit_predict_binary(
                        m_name, X_tr, y_dev[tr], w_tr,
                        dev_sub.iloc[tr]["participant_id"].to_numpy(), X_va
                    )
                    oof_p[va] = p_va; oof_thr[va] = thr
                met_dev = evaluate_binary_full(
                    y_dev, oof_p, oof_thr, dev_sub["episode_id"].to_numpy(), dev_sub["participant_id"].to_numpy()
                )
                if met_dev["ep_auc"] > best_dev_auc:
                    best_dev_auc = met_dev["ep_auc"]
                    best_dev_row = {"model": m_name, "features": feat, "normalization": norm, "dev_lopo": met_dev}

    # Fit sekali pada full Dev -> evaluasi sekali pada Held-Out Test
    feat_w = best_dev_row["features"]
    norm_w = best_dev_row["normalization"]
    m_w = best_dev_row["model"]
    prep_w = RobustPreprocessor(FEATURE_PROFILES[feat_w], norm_w).fit(dev_sub)
    w_dev = episode_balanced_weights(dev_sub, y_dev)
    p_te, thr_te = fit_predict_binary(
        m_w, prep_w.transform(dev_sub), y_dev, w_dev,
        dev_sub["participant_id"].to_numpy(), prep_w.transform(te_sub)
    )
    met_te = evaluate_binary_full(
        y_te, p_te, np.full(len(y_te), thr_te), te_sub["episode_id"].to_numpy(), te_sub["participant_id"].to_numpy()
    )
    dev_test_summary[task_name] = {
        "locked_candidate": f"{m_w}|{feat_w}|{norm_w}",
        "dev_lopo": best_dev_row["dev_lopo"],
        "held_out_test": met_te,
    }
    print(f"\n[{task_name}] Locked from 13-person Dev LOPO: {m_w}|{feat_w}|{norm_w}")
    print(f"  Dev LOPO (13p) : win_AUC={best_dev_row['dev_lopo']['win_auc']:.3f}, ep_AUC={best_dev_row['dev_lopo']['ep_auc']:.3f}, ep_BalAcc={best_dev_row['dev_lopo']['ep_bal_acc']:.2%}, ep_Acc={best_dev_row['dev_lopo']['ep_acc']:.2%}")
    print(f"  Held-Out (4p)  : win_AUC={met_te['win_auc']:.3f}, ep_AUC={met_te['ep_auc']:.3f}, ep_BalAcc={met_te['ep_bal_acc']:.2%}, ep_Acc={met_te['ep_acc']:.2%}, sens={met_te['ep_sens']:.2%}, spec={met_te['ep_spec']:.2%}")

# --- Audit 1: Koefisien LogReg_L2 terstandardisasi & stabilitas tanda lintas 17 fold LOPO ---
coef_rows = []
for task_name, tcfg in BINARY_TASKS.items():
    sub = df[df["difficulty_rating"].isin(tcfg["low_vals"] | tcfg["high_vals"])].reset_index(drop=True)
    y = sub["difficulty_rating"].isin(tcfg["high_vals"]).astype(int).to_numpy()
    cols = FEATURE_PROFILES["core9_head4_base"]
    fold_coefs = []
    for tr, _ in LeaveOneGroupOut().split(sub, y, sub["participant_id"].to_numpy()):
        prep = RobustPreprocessor(cols, "global_robust").fit(sub.iloc[tr])
        X_tr = prep.transform(sub.iloc[tr])
        w_tr = episode_balanced_weights(sub.iloc[tr], y[tr])
        lr = LogisticRegression(C=0.5, max_iter=5000, random_state=SEED).fit(X_tr, y[tr], sample_weight=w_tr)
        fold_coefs.append(lr.coef_[0])
    fc = np.array(fold_coefs)
    for j, c in enumerate(cols):
        coef_rows.append({
            "task": task_name, "feature": c,
            "coef_mean": float(fc[:, j].mean()),
            "coef_std": float(fc[:, j].std()),
            "abs_mean": float(abs(fc[:, j].mean())),
            "pos_folds": int((fc[:, j] > 0).sum()),
            "n_folds": int(len(fc)),
        })
coef_df = pd.DataFrame(coef_rows).sort_values(["task", "abs_mean"], ascending=[True, False]).reset_index(drop=True)
coef_df.to_csv(OUTPUT_DIR / "feature_coefficients_lopo.csv", index=False)

print("\n=== Top 8 Koefisien Stabil (LogReg_L2 | core9_head4_base | global_robust) pada extreme_1_3_vs_7_10 ===")
print(coef_df[coef_df["task"] == "extreme_1_3_vs_7_10"][["feature", "coef_mean", "coef_std", "pos_folds", "n_folds"]].head(8).to_string(index=False))

# --- Audit 2: Faktor Perancu (Confounding Check) ---
# Cek apakah probabilitas LOPO tetap memisahkan Low vs High:
#   (a) hanya pada baris di mana rmssd_baseline_ratio TIDAK missing,
#   (b) di dalam setiap difficulty_level protokol (easy/medium/hard),
#   (c) korelasi parsial terhadap rating setelah mengontrol waktu sesi (session_elapsed_min).
confound_audit = {}
for task_name in BINARY_TASKS:
    cid = "LogReg_L2|core9_head4_base|global_robust"
    sub, y, oof_p, oof_thr = oof_store[(task_name, cid)]
    ep_df = pd.DataFrame({
        "episode_id": sub["episode_id"],
        "participant_id": sub["participant_id"],
        "difficulty_level": sub["difficulty_level"],
        "elapsed_min": sub["session_elapsed_min"],
        "has_baseline": sub["rmssd_baseline_ratio"].notna().astype(int),
        "y": y,
        "p": oof_p,
    }).groupby(["episode_id", "participant_id", "difficulty_level"], as_index=False).agg(
        elapsed_min=("elapsed_min", "median"),
        has_baseline=("has_baseline", "min"),
        y=("y", "first"),
        p=("p", "mean"),
    )
    # 1. Tanpa baris missing baseline
    non_miss = ep_df[ep_df["has_baseline"] == 1]
    auc_no_miss = float(roc_auc_score(non_miss["y"], non_miss["p"])) if non_miss["y"].nunique() == 2 else np.nan
    # 2. Dalam setiap difficulty_level protokol
    by_diff = {}
    for d in ("easy", "medium", "hard"):
        g = ep_df[ep_df["difficulty_level"] == d]
        if g["y"].nunique() == 2 and len(g) >= 5:
            by_diff[d] = {"n_ep": int(len(g)), "n_high": int(g["y"].sum()), "ep_auc": float(roc_auc_score(g["y"], g["p"]))}
    # 3. Korelasi dengan waktu sesi vs korelasi parsial (residualize p dan y terhadap elapsed_min)
    rho_time = float(stats.spearmanr(ep_df["p"], ep_df["elapsed_min"]).statistic)
    # Linear residualization terhadap elapsed_min
    t_arr = ep_df["elapsed_min"].to_numpy()
    p_res = ep_df["p"].to_numpy() - np.polyval(np.polyfit(t_arr, ep_df["p"].to_numpy(), 1), t_arr)
    y_res = ep_df["y"].to_numpy() - np.polyval(np.polyfit(t_arr, ep_df["y"].to_numpy(), 1), t_arr)
    part_r, part_p = stats.pearsonr(p_res, y_res)
    confound_audit[task_name] = {
        "full_ep_auc": float(roc_auc_score(ep_df["y"], ep_df["p"])),
        "non_missing_baseline_ep_auc": auc_no_miss,
        "within_protocol_difficulty": by_diff,
        "spearman_pred_vs_elapsed_min": rho_time,
        "partial_r_controlling_elapsed_min": float(part_r),
        "partial_p_controlling_elapsed_min": float(part_p),
    }

print("\n=== AUDIT FAKTOR PERANCU (LogReg_L2 | core9_head4_base | global_robust) ===")
print(json.dumps(confound_audit, indent=2))

# --- Audit 3: Perilaku Zona Netral / Medium (rating 4–6) pada Model extreme_1_3_vs_7_10 ---
# Untuk setiap peserta yang ditahan (LOPO), latih pada ekstrem (1-3 vs 7-10) peserta lain,
# lalu prediksi p(High) pada seluruh episode peserta uji (termasuk 4-6 yang tidak pernah masuk training).
ext_cols = FEATURE_PROFILES["core9_head4_base"]
neutral_ep_rows = []
for hold_pid in sorted(df["participant_id"].unique()):
    tr_all = df[(df["participant_id"] != hold_pid) & (df["difficulty_rating"].isin({1, 2, 3, 7, 8, 9, 10}))].reset_index(drop=True)
    va_all = df[df["participant_id"] == hold_pid].reset_index(drop=True)
    y_tr = tr_all["difficulty_rating"].isin({7, 8, 9, 10}).astype(int).to_numpy()
    prep = RobustPreprocessor(ext_cols, "global_robust").fit(tr_all)
    w_tr = episode_balanced_weights(tr_all, y_tr)
    p_va, _ = fit_predict_binary("LogReg_L2", prep.transform(tr_all), y_tr, w_tr, tr_all["participant_id"].to_numpy(), prep.transform(va_all))
    tmp = pd.DataFrame({
        "episode_id": va_all["episode_id"], "participant_id": hold_pid,
        "rating": va_all["difficulty_rating"], "p_high": p_va,
    }).groupby(["episode_id", "participant_id", "rating"], as_index=False)["p_high"].mean()
    neutral_ep_rows.append(tmp)

neutral_ep_df = pd.concat(neutral_ep_rows, ignore_index=True)
neutral_ep_df["band"] = pd.cut(neutral_ep_df["rating"], bins=[0, 3, 6, 10], labels=["Low (1-3)", "Medium (4-6)", "High (7-10)"]).astype(str)
neutral_ep_df.to_csv(OUTPUT_DIR / "neutral_zone_episode_probs.csv", index=False)

neutral_audit = {}
print("\n=== AUDIT ZONA NETRAL MEDIUM (4–6) PADA MODEL EXTREME (1–3 vs 7–10) ===")
for b in ["Low (1-3)", "Medium (4-6)", "High (7-10)"]:
    g = neutral_ep_df[neutral_ep_df["band"] == b]["p_high"].to_numpy()
    neutral_audit[b] = {
        "n_episodes": int(len(g)),
        "median_p": float(np.median(g)),
        "mean_p": float(np.mean(g)),
        "q25_p": float(np.percentile(g, 25)),
        "q75_p": float(np.percentile(g, 75)),
        "frac_lt_035": float((g < 0.35).mean()),
        "frac_035_065": float(((g >= 0.35) & (g <= 0.65)).mean()),
        "frac_gt_050": float((g >= 0.50).mean()),
        "frac_gt_065": float((g > 0.65).mean()),
    }
    r = neutral_audit[b]
    print(f"  {b:12s} (n={r['n_episodes']:2d}ep): median p={r['median_p']:.3f} [IQR {r['q25_p']:.3f}–{r['q75_p']:.3f}] | <0.35: {r['frac_lt_035']:.1%}, 0.35–0.65: {r['frac_035_065']:.1%}, >0.65: {r['frac_gt_065']:.1%} (>=0.50: {r['frac_gt_050']:.1%})")



[extreme_1_3_vs_7_10] Locked from 13-person Dev LOPO: Blend_LR_ET|core9_head4|causal_session_robust
  Dev LOPO (13p) : win_AUC=0.586, ep_AUC=0.735, ep_BalAcc=73.50%, ep_Acc=71.43%
  Held-Out (4p)  : win_AUC=0.654, ep_AUC=0.629, ep_BalAcc=58.57%, ep_Acc=57.89%, sens=60.00%, spec=57.14%

[full_1_5_vs_6_10] Locked from 13-person Dev LOPO: ExtraTrees_Shallow|core9|causal_session_robust
  Dev LOPO (13p) : win_AUC=0.549, ep_AUC=0.620, ep_BalAcc=63.19%, ep_Acc=56.57%
  Held-Out (4p)  : win_AUC=0.670, ep_AUC=0.693, ep_BalAcc=65.71%, ep_Acc=62.96%, sens=71.43%, spec=60.00%

[full_1_6_vs_7_10] Locked from 13-person Dev LOPO: LogReg_L2|core9|global_robust
  Dev LOPO (13p) : win_AUC=0.724, ep_AUC=0.739, ep_BalAcc=75.31%, ep_Acc=66.67%
  Held-Out (4p)  : win_AUC=0.446, ep_AUC=0.773, ep_BalAcc=60.45%, ep_Acc=48.15%, sens=80.00%, spec=40.91%

=== Top 8 Koefisien Stabil (LogReg_L2 | core9_head4_base | global_robust) pada extreme_1_3_vs_7_10 ===
                feature  coef_mean  coef_std  pos_folds 

In [6]:
# Cell 6: Visualisasi Publikasi & Simpan Semua Artefak (JSON, CSV, Figures, RESULTS.md)
# 1. Plot ROC Curves (Episode-Level LOPO) untuk ketiga skema biner + perbandingan grup fitur
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

# Panel 1: ROC Curves LOPO Episode untuk 3 skema biner (LogReg_L2 | core9_head4_base | global_robust)
colors_t = {"extreme_1_3_vs_7_10": "#2563eb", "full_1_6_vs_7_10": "#10b981", "full_1_5_vs_6_10": "#f59e0b"}
for tname in BINARY_TASKS:
    cid = "LogReg_L2|core9_head4_base|global_robust"
    sub, y, oof_p, _ = oof_store[(tname, cid)]
    ep_d = pd.DataFrame({"ep": sub["episode_id"], "y": y, "p": oof_p}).groupby("ep", as_index=False).mean()
    ep_y = ep_d["y"].round().astype(int).to_numpy()
    ep_p = ep_d["p"].to_numpy()
    fpr, tpr, _ = roc_curve(ep_y, ep_p)
    auc = roc_auc_score(ep_y, ep_p)
    axes[0].plot(fpr, tpr, label=f"{tname} (AUC={auc:.3f})", color=colors_t[tname], linewidth=2)
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random (0.500)")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("LOPO 17-Peserta: Episode ROC Curve")
axes[0].legend(fontsize=7.5, loc="lower right")

# Panel 2: Ablasi Grup Fitur pada extreme_1_3_vs_7_10 & full_1_6_vs_7_10 (LogReg_L2 | global_robust)
feat_order = ["only_hrv2", "only_eye7", "only_head4", "only_base1", "only_head4_base1", "core9", "core9_head4", "core9_head4_base", "core9_head4_base_gaze"]
sub_f1 = lopo_df[(lopo_df["task"] == "extreme_1_3_vs_7_10") & (lopo_df["model"] == "LogReg_L2") & (lopo_df["normalization"] == "global_robust")].set_index("features").reindex(feat_order)
sub_f2 = lopo_df[(lopo_df["task"] == "full_1_6_vs_7_10") & (lopo_df["model"] == "LogReg_L2") & (lopo_df["normalization"] == "global_robust")].set_index("features").reindex(feat_order)
ypos = np.arange(len(feat_order))
axes[1].barh(ypos - 0.18, sub_f1["ep_auc"], height=0.35, label="extreme 1-3 vs 7-10", color="#2563eb", alpha=0.85)
axes[1].barh(ypos + 0.18, sub_f2["ep_auc"], height=0.35, label="full 1-6 vs 7-10", color="#10b981", alpha=0.85)
axes[1].axvline(0.5, color="red", linestyle="--", linewidth=1)
axes[1].set_yticks(ypos); axes[1].set_yticklabels(feat_order, fontsize=8)
axes[1].set_xlim(0.3, 0.85); axes[1].set_xlabel("LOPO Episode ROC-AUC")
axes[1].set_title("Ablasi Grup Fitur (LogReg L2, Global Robust)")
axes[1].legend(fontsize=7.5, loc="lower right")

# Panel 3: Koefisien fitur terstandardisasi (extreme_1_3_vs_7_10)
c_ext = coef_df[coef_df["task"] == "extreme_1_3_vs_7_10"].sort_values("coef_mean")
cols_bar = ["#ef4444" if v > 0 else "#3b82f6" for v in c_ext["coef_mean"]]
axes[2].barh(c_ext["feature"], c_ext["coef_mean"], xerr=c_ext["coef_std"], color=cols_bar, alpha=0.85, capsize=2)
axes[2].axvline(0.0, color="black", linewidth=1)
axes[2].set_xlabel("LogReg Coef (+ = lebih tinggi ke High)")
axes[2].set_title("Arah & Stabilitas Koefisien (17 Fold LOPO)")
axes[2].tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "v4_binary_deepdive_overview.png", dpi=150)
plt.close(fig)

# 2. Plot Performa per Peserta (Episode-level accuracy / score pada extreme_1_3_vs_7_10 & full_1_6_vs_7_10)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, tname in zip(axes, ["extreme_1_3_vs_7_10", "full_1_6_vs_7_10"]):
    cid = "LogReg_L2|core9_head4_base|global_robust"
    sub, y, oof_p, oof_thr = oof_store[(tname, cid)]
    ep_d = pd.DataFrame({
        "ep": sub["episode_id"], "pid": sub["participant_id"], "y": y, "p": oof_p
    }).groupby(["ep", "pid"], as_index=False).mean()
    ep_d["y"] = ep_d["y"].round().astype(int)
    ep_d["pred"] = (ep_d["p"] >= 0.5).astype(int)
    cm = confusion_matrix(ep_d["y"], ep_d["pred"], labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    lbls = ["Low (1-3)", "High (7-10)"] if "1_3" in tname else ["Non-Overload (1-6)", "Overload (7-10)"]
    ax.set_xticks([0, 1]); ax.set_xticklabels(lbls, fontsize=9)
    ax.set_yticks([0, 1]); ax.set_yticklabels(lbls, fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    auc = roc_auc_score(ep_d["y"], ep_d["p"])
    bal = balanced_accuracy_score(ep_d["y"], ep_d["pred"])
    ax.set_title(f"{tname}\nLOPO Episode CM (n={len(ep_d)}, AUC={auc:.3f}, BalAcc={bal:.1%})")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "v4_binary_lopo_confusion_matrices.png", dpi=150)
plt.close(fig)

# Simpan ringkasan JSON & RESULTS.md
best_lopo_per_task = {}
for tname in BINARY_TASKS:
    sub_lb = lopo_df[(lopo_df["task"] == tname) & (~lopo_df["is_single_group_ablation"]) & (lopo_df["normalization"] != "unsupervised_session_robust")]
    top_r = sub_lb.iloc[0]
    best_lopo_per_task[tname] = {
        k: (None if isinstance(v, float) and np.isnan(v) else float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else bool(v) if isinstance(v, (np.bool_, bool)) else v)
        for k, v in top_r.to_dict().items()
    }

payload = {
    "experiment": "batch-06/exp03_v4_binary_overload",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_sha256": file_hashes,
    "best_lopo_operational_per_task": best_lopo_per_task,
    "dev_to_heldout_test_summary": dev_test_summary,
    "confound_audit_core9_head4_base": confound_audit,
    "neutral_zone_audit_extreme_model": neutral_audit,
}
(OUTPUT_DIR / "binary_deepdive_summary.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

md = [
    "# Hasil Eksperimen Batch 06 — Exp03: Deep-Dive Deteksi Biner Overload (`v4_self_reported_rating`)",
    "",
    f"- **Tanggal run (UTC):** {payload['generated_at_utc']}",
    "- **Evaluasi utama:** Leave-One-Participant-Out (LOPO) pada seluruh 17 peserta + cluster-bootstrap 95% CI + verifikasi silang pada Held-Out Test beku Exp02 (13 Dev -> 4 Test).",
    "",
    "## 1. Perbandingan 3 Skema Biner (LOPO 17 Peserta — Kandidat Operasional Terbaik)",
    "",
    "| Skema Biner | Model Terbaik (Operasional) | Win AUC | Ep AUC (95% CI) | Ep BalAcc | Ep Acc | Ep Sens / Spec | MWU p-value |",
    "|---|---|---:|---|---:|---:|---:|---:|",
]
for tname, r in best_lopo_per_task.items():
    ci_str = f"{r['ep_auc']:.3f} ({r['ep_auc_ci95_lo']:.3f}–{r['ep_auc_ci95_hi']:.3f})" if r.get("ep_auc_ci95_lo") is not None else f"{r['ep_auc']:.3f}"
    md.append(
        f"| `{tname}` | `{r['candidate_id']}` | {r['win_auc']:.3f} | **{ci_str}** | {r['ep_bal_acc']:.2%} | {r['ep_acc']:.2%} | {r['ep_sens']:.1%} / {r['ep_spec']:.1%} | {r['ep_mwu_pvalue']:.4f} |"
    )

md += [
    "",
    "## 2. Verifikasi Silang pada Held-Out Test Beku (13 Dev -> 4 Test)",
    "",
    "| Skema Biner | Model Terkunci (dari Dev LOPO) | Dev LOPO Ep AUC | Test Ep AUC | Test Ep BalAcc | Test Ep Acc |",
    "|---|---|---:|---:|---:|---:|",
]
for tname, d in dev_test_summary.items():
    md.append(
        f"| `{tname}` | `{d['locked_candidate']}` | {d['dev_lopo']['ep_auc']:.3f} | **{d['held_out_test']['ep_auc']:.3f}** | {d['held_out_test']['ep_bal_acc']:.2%} | {d['held_out_test']['ep_acc']:.2%} |"
    )

md += [
    "",
    "## 3. Audit Faktor Perancu (`LogReg_L2 | core9_head4_base | global_robust`)",
    "",
    "| Skema Biner | Full Ep AUC | Non-Missing Baseline Ep AUC | Spearman(pred, waktu sesi) | Partial $r$ (kontrol waktu) | Partial $p$ |",
    "|---|---:|---:|---:|---:|---:|",
]
for tname, ca in confound_audit.items():
    md.append(
        f"| `{tname}` | {ca['full_ep_auc']:.3f} | {ca['non_missing_baseline_ep_auc']:.3f} | {ca['spearman_pred_vs_elapsed_min']:+.3f} | {ca['partial_r_controlling_elapsed_min']:+.3f} | {ca['partial_p_controlling_elapsed_min']:.4f} |"
    )

md += [
    "",
    "![V4 Binary Deep-Dive Overview](figures/v4_binary_deepdive_overview.png)",
    "",
    "![V4 Binary LOPO Confusion Matrices](figures/v4_binary_lopo_confusion_matrices.png)",
]
(OUTPUT_DIR / "RESULTS.md").write_text("\n".join(md) + "\n", encoding="utf-8")
print("Semua artefak Exp03 tersimpan di:", OUTPUT_DIR)


Semua artefak Exp03 tersimpan di: E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\outputs\exp03_v4_binary_overload
